In [6]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv('connections_data.csv')
df['commodity_list'] = df['commodity'].apply(
    lambda x: [c.strip() for c in x.split(';')]
)
print(df.shape)
df.head()

(100, 8)


,user_id,commodity,role,city,state,min_quantity_mt,max_quantity_mt,commodity_list
0,1,rice,trader,Nagpur,Maharashtra,50,200,[rice]
1,2,sugar,broker,Nagpur,Maharashtra,20,100,[sugar]
2,3,cotton,exporter,Nagpur,Maharashtra,100,500,[cotton]
3,4,rice;sugar,trader,Nagpur,Maharashtra,30,150,"[rice, sugar]"
4,5,cotton;sugar,broker,Nagpur,Maharashtra,40,180,"[cotton, sugar]"


In [59]:
ALL_COMMODITIES = sorted(set(
    c for lst in df['commodity_list'] for c in lst
))
print("Commodities found:", ALL_COMMODITIES)

COMMODITY_BOOST = 1.0

def encode_commodity(commodity_list):
    vec = np.zeros(len(ALL_COMMODITIES))
    for c in commodity_list:
        if c in ALL_COMMODITIES:
            vec[ALL_COMMODITIES.index(c)] = 1.0
    return vec * COMMODITY_BOOST

commodity_vecs = np.array([
    encode_commodity(row['commodity_list'])
    for _, row in df.iterrows()
])

print("Dimension names:", [f"has_{c}" for c in ALL_COMMODITIES])
print()
for i in range(5):
    row = df.iloc[i]
    raw = commodity_vecs[i] / COMMODITY_BOOST
    vals = dict(zip([f"has_{c}" for c in ALL_COMMODITIES], raw))
    print(f"User {row['user_id']} | {row['commodity']:20s} → {vals}")

Commodities found: ['cotton', 'rice', 'sugar']
Dimension names: ['has_cotton', 'has_rice', 'has_sugar']

User 1 | rice                 → {'has_cotton': np.float64(0.0), 'has_rice': np.float64(1.0), 'has_sugar': np.float64(0.0)}
User 2 | sugar                → {'has_cotton': np.float64(0.0), 'has_rice': np.float64(0.0), 'has_sugar': np.float64(1.0)}
User 3 | cotton               → {'has_cotton': np.float64(1.0), 'has_rice': np.float64(0.0), 'has_sugar': np.float64(0.0)}
User 4 | rice;sugar           → {'has_cotton': np.float64(0.0), 'has_rice': np.float64(1.0), 'has_sugar': np.float64(1.0)}
User 5 | cotton;sugar         → {'has_cotton': np.float64(1.0), 'has_rice': np.float64(0.0), 'has_sugar': np.float64(1.0)}


In [66]:
ROLE_AFFINITY = {
    'trader':   {'want_broker': 0.5, 'want_exporter': 0.3, 'want_trader': 0.2},
    'broker':   {'want_broker': 0.33,'want_exporter': 0.33,'want_trader': 0.33},
    'exporter': {'want_broker': 0.5, 'want_exporter': 0.2, 'want_trader': 0.3},
}

# What each role OFFERS — used for candidate vectors
ROLE_OFFERS = {
    'trader':   {'want_broker': 0.0, 'want_exporter': 0.0, 'want_trader': 1.0},
    'broker':   {'want_broker': 1.0, 'want_exporter': 0.0, 'want_trader': 0.0},
    'exporter': {'want_broker': 0.0, 'want_exporter': 1.0, 'want_trader': 0.0},
}

ROLE_DIMS  = ['want_broker', 'want_exporter', 'want_trader']
ROLE_BOOST = 1.5

def encode_role_searcher(role):
    """What this user WANTS — used for the query vector."""
    affinity = ROLE_AFFINITY[role]
    return np.array([affinity[d] for d in ROLE_DIMS]) * ROLE_BOOST

def encode_role_candidate(role):
    """What this user IS — used for stored vectors in DB."""
    offers = ROLE_OFFERS[role]
    return np.array([offers[d] for d in ROLE_DIMS]) * ROLE_BOOST

# Stored vectors use candidate encoding (what they ARE)
role_vecs = np.array([
    encode_role_candidate(row['role'])
    for _, row in df.iterrows()
])

print("Dimension names:", ROLE_DIMS)
print()
print("Candidate vectors (what they ARE — stored in DB):")
for role in ['trader', 'broker', 'exporter']:
    v = encode_role_candidate(role) / ROLE_BOOST
    print(f"  {role:10s} → {dict(zip(ROLE_DIMS, v))}")

print()
print("Searcher vectors (what they WANT — used at query time only):")
for role in ['trader', 'broker', 'exporter']:
    v = encode_role_searcher(role) / ROLE_BOOST
    print(f"  {role:10s} → {dict(zip(ROLE_DIMS, v))}")

print()
print("Dot product preview for trader searching:")
trader_want = encode_role_searcher('trader')
for role in ['broker', 'exporter', 'trader']:
    candidate_offer = encode_role_candidate(role)
    dot = np.dot(trader_want, candidate_offer)
    print(f"  trader → {role}: dot = {dot:.3f}")

Dimension names: ['want_broker', 'want_exporter', 'want_trader']

Candidate vectors (what they ARE — stored in DB):
  trader     → {'want_broker': np.float64(0.0), 'want_exporter': np.float64(0.0), 'want_trader': np.float64(1.0)}
  broker     → {'want_broker': np.float64(1.0), 'want_exporter': np.float64(0.0), 'want_trader': np.float64(0.0)}
  exporter   → {'want_broker': np.float64(0.0), 'want_exporter': np.float64(1.0), 'want_trader': np.float64(0.0)}

Searcher vectors (what they WANT — used at query time only):
  trader     → {'want_broker': np.float64(0.5), 'want_exporter': np.float64(0.3), 'want_trader': np.float64(0.20000000000000004)}
  broker     → {'want_broker': np.float64(0.33), 'want_exporter': np.float64(0.33), 'want_trader': np.float64(0.33)}
  exporter   → {'want_broker': np.float64(0.5), 'want_exporter': np.float64(0.20000000000000004), 'want_trader': np.float64(0.3)}

Dot product preview for trader searching:
  trader → broker: dot = 1.125
  trader → exporter: dot = 0.

In [67]:
city_enc  = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
state_enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

city_vecs  = city_enc.fit_transform(df[['city']])
state_vecs = state_enc.fit_transform(df[['state']])

print("City dimensions:",  city_enc.categories_[0].tolist())
print("State dimensions:", state_enc.categories_[0].tolist())
print()
print("City vec shape:", city_vecs.shape)
print("State vec shape:", state_vecs.shape)

City dimensions: ['Agartala', 'Ahmedabad', 'Ahmednagar', 'Aizawl', 'Akola', 'Alibaug', 'Amravati', 'Amritsar', 'Aurangabad', 'Bangalore', 'Beed', 'Bhandara', 'Bhopal', 'Bhubaneswar', 'Bilaspur', 'Buldhana', 'Chandigarh', 'Chandrapur', 'Chennai', 'Coimbatore', 'Cuttack', 'Daman', 'Dehradun', 'Delhi', 'Dhule', 'Dombivli', 'Gadchiroli', 'Gangtok', 'Gondia', 'Guwahati', 'Haridwar', 'Hingoli', 'Howrah', 'Hyderabad', 'Imphal', 'Indore', 'Itanagar', 'Jaipur', 'Jalgaon', 'Jammu', 'Kalyan', 'Kanpur', 'Kargil', 'Kavaratti', 'Kochi', 'Kohima', 'Kolhapur', 'Kolkata', 'Latur', 'Leh', 'Lucknow', 'Ludhiana', 'Madurai', 'Margao', 'Mumbai', 'Mysore', 'Nagpur', 'Nanded', 'Nashik', 'Navi Mumbai', 'Noida', 'Osmanabad', 'Palghar', 'Panaji', 'Panvel', 'Parbhani', 'Patna', 'Port Blair', 'Pune', 'Raipur', 'Ranchi', 'Ratnagiri', 'Sangli', 'Satara', 'Shillong', 'Shimla', 'Silvassa', 'Sindhudurg', 'Solan', 'Solapur', 'Srinagar', 'Surat', 'Thane', 'Trivandrum', 'Udaipur', 'Ulhasnagar', 'Virar', 'Warangal', 'Wardh

In [68]:
# Final vector layout:
# [ has_cotton | has_rice | has_sugar | want_broker | want_exporter | want_trader | city_onehot... | state_onehot... ]

master_vectors = np.hstack([
    commodity_vecs,   # explicitly named has_X dims
    role_vecs,        # explicitly named want_X dims
    # city_vecs,
    state_vecs,
])

df['vector'] = master_vectors.tolist()

# Print dimension layout so you know exactly what each position means
n_comm  = commodity_vecs.shape[1]
n_role  = role_vecs.shape[1]
# n_city  = city_vecs.shape[1]
n_state = state_vecs.shape[1]
total   = master_vectors.shape[1]

print(f"Total vector dims: {total}")
print(f"  [0 : {n_comm}]              → commodity  ({[f'has_{c}' for c in ALL_COMMODITIES]})")
print(f"  [{n_comm} : {n_comm+n_role}]          → role       ({ROLE_DIMS})")
print(f"  [{n_comm+n_role} : {n_comm+n_role+n_city}]   → city       ({city_enc.categories_[0].tolist()})")
print(f"  [{n_comm+n_role+n_city} : {total}]  → state      ({state_enc.categories_[0].tolist()})")

Total vector dims: 40
  [0 : 3]              → commodity  (['has_cotton', 'has_rice', 'has_sugar'])
  [3 : 6]          → role       (['want_broker', 'want_exporter', 'want_trader'])
  [6 : 97]   → city       (['Agartala', 'Ahmedabad', 'Ahmednagar', 'Aizawl', 'Akola', 'Alibaug', 'Amravati', 'Amritsar', 'Aurangabad', 'Bangalore', 'Beed', 'Bhandara', 'Bhopal', 'Bhubaneswar', 'Bilaspur', 'Buldhana', 'Chandigarh', 'Chandrapur', 'Chennai', 'Coimbatore', 'Cuttack', 'Daman', 'Dehradun', 'Delhi', 'Dhule', 'Dombivli', 'Gadchiroli', 'Gangtok', 'Gondia', 'Guwahati', 'Haridwar', 'Hingoli', 'Howrah', 'Hyderabad', 'Imphal', 'Indore', 'Itanagar', 'Jaipur', 'Jalgaon', 'Jammu', 'Kalyan', 'Kanpur', 'Kargil', 'Kavaratti', 'Kochi', 'Kohima', 'Kolhapur', 'Kolkata', 'Latur', 'Leh', 'Lucknow', 'Ludhiana', 'Madurai', 'Margao', 'Mumbai', 'Mysore', 'Nagpur', 'Nanded', 'Nashik', 'Navi Mumbai', 'Noida', 'Osmanabad', 'Palghar', 'Panaji', 'Panvel', 'Parbhani', 'Patna', 'Port Blair', 'Pune', 'Raipur', 'Ranchi', 'Ratn

In [69]:
# Quantity stored as raw metadata only — used for DB filter, not in vector
vector_db = []

for idx, row in df.iterrows():
    vector_db.append({
        "id":     int(row['user_id']),
        "vector": master_vectors[idx],
        "meta": {
            "user_id":        int(row['user_id']),
            "role":           row['role'],
            "commodity":      row['commodity'],
            "commodity_list": row['commodity_list'],
            "city":           row['city'],
            "state":          row['state'],
            "qty_min":        int(row['min_quantity_mt']),
            "qty_max":        int(row['max_quantity_mt']),
        }
    })

print(f"Vector DB: {len(vector_db)} records")
print("\nSample:")
r = vector_db[0]
print(f"  id: {r['id']}")
print(f"  meta: {r['meta']}")
print(f"  vector dims: {len(r['vector'])}")
print(f"  vector (commodity+role section): {r['vector'][:n_comm+n_role]}")
print(f"  names: {[f'has_{c}' for c in ALL_COMMODITIES] + ROLE_DIMS}")

Vector DB: 100 records

Sample:
  id: 1
  meta: {'user_id': 1, 'role': 'trader', 'commodity': 'rice', 'commodity_list': ['rice'], 'city': 'Nagpur', 'state': 'Maharashtra', 'qty_min': 50, 'qty_max': 200}
  vector dims: 40
  vector (commodity+role section): [0.  1.  0.  0.  0.  1.5]
  names: ['has_cotton', 'has_rice', 'has_sugar', 'want_broker', 'want_exporter', 'want_trader']


In [70]:
def search(searcher_id, top_k=20):
    searcher = next(r for r in vector_db if r['meta']['user_id'] == searcher_id)
    searcher_role = searcher['meta']['role']

    # Build query vector with WANT encoding — not what's stored in DB
    s_commodity = encode_commodity(searcher['meta']['commodity_list'])
    s_role      = encode_role_searcher(searcher_role)   # WANT dims
    s_state     = state_enc.transform([[searcher['meta']['state']]])[0]

    s_vec = np.concatenate([s_commodity, s_role, s_state]).reshape(1, -1)

    # All users except searcher
    candidates = [
        r for r in vector_db
        if r['meta']['user_id'] != searcher_id
    ]

    if not candidates:
        print("No candidates found.")
        return pd.DataFrame()

    # Stored vectors use IS (offer) encoding — query uses WANT encoding
    # Dot product is now high when want_broker × is_broker = 0.5×4 × 1.0×4 = 8.0
    # vs want_trader × is_trader = 0.2×4 × 1.0×4 = 3.2
    cand_vecs = np.array([r['vector'] for r in candidates])
    sims = cosine_similarity(s_vec, cand_vecs)[0]

    top_idx = np.argsort(sims)[::-1][:top_k]

    results = []
    for i in top_idx:
        m = candidates[i]['meta']
        results.append({
            'user_id':    m['user_id'],
            'role':       m['role'],
            'commodity':  m['commodity'],
            'city':       m['city'],
            'state':      m['state'],
            'qty_range':  f"{m['qty_min']}–{m['qty_max']}mt",
            'similarity': round(float(sims[i]), 4),
        })

    return pd.DataFrame(results)


In [71]:
TEST_USER_ID = 4

s = next(r['meta'] for r in vector_db if r['meta']['user_id'] == TEST_USER_ID)
print(f"Searcher: User {s['user_id']} | {s['role']} | {s['commodity']} | {s['city']} | {s['qty_min']}–{s['qty_max']}mt")
print("=" * 70)

results = search(TEST_USER_ID, top_k=15)
print(results.to_string(index=False))

print("\nWhat to verify:")
print("1. Same commodity users rank highest")
print("2. For a trader: brokers outrank exporters, exporters outrank traders")
print("3. Same city users rank above different city (same commodity+role)")
print("4. Anyone with zero qty overlap is absent from results entirely")

Searcher: User 4 | trader | rice;sugar | Nagpur | 30–150mt
 user_id   role    commodity        city       state qty_range  similarity
      98 broker        sugar     Alibaug Maharashtra  20–100mt      0.7720
      88 broker        sugar      Washim Maharashtra  20–100mt      0.7720
      92 broker        sugar Navi Mumbai Maharashtra  25–120mt      0.7720
      72 broker        sugar     Jalgaon Maharashtra  30–150mt      0.7720
      68 broker        sugar    Kolhapur Maharashtra  20–100mt      0.7720
      82 broker        sugar  Gadchiroli Maharashtra  25–120mt      0.7720
      78 broker        sugar    Yavatmal Maharashtra  20–100mt      0.7720
      12 broker        sugar        Pune Maharashtra  20–100mt      0.7720
       8 broker        sugar      Nagpur Maharashtra  25–110mt      0.7720
       2 broker        sugar      Nagpur Maharashtra  20–100mt      0.7720
      74 trader   rice;sugar       Latur Maharashtra  45–170mt      0.7669
      84 trader   rice;sugar        Beed 

c:\Users\Admin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
